# Ch 2: Preprocessing Text Data  

- Machines are not like humans, that they will not understand your feelings and emotions in words.
- Machines understands in terms numbers and performs operations on it to do some task.
- So, in the very first step of building an LLM we need to transform words and sentences into some numbers, to make it suitable for training to understand the context and generate text.
- For preprocessing the text, we will go through the following steps:
    1. Tokenization
    2. Creating Dataset for training
    3. Embedding tokens


## Tokenization using Byte Pair Encoding(BPE)
- **Tokenization** is the method of decomposing a sentence into a set of **tokens** and assigning unique ids to them.
- **Tokens** are basic building blocks of a sentence,which are basically words but it can be symbol,number,sub-word,etc. depending upon the tokenizer user.
- During tokenising we face 2 issues:
    1. **Unknown tokens:** These are the tokens out of the corpus used as dataset. These tokens need to be uniquely identified.
    2. **Special tokens:** These are the tokens used for special purposes.Commonly used special tokens are beginning of sequence(BOS), end of sequence(EOS) and padding(PAD). 
- Most commonly used tokenizer in LLMs is **Byte Pair Encoding(BPE)**.
- It solves the issue of identifying unknown tokens, by break the unknown word into a particular set of subtokens and assigning unique tokens to them as shown below.

<p align="center"><img src="Images/Screenshot 2025-06-16 124239.png" width="450" height=""></p>

- We will learn in detail about BPE later by making a BPE from scratch. For now, we will use `tiktoken` for implementing BPE.
- **Tiktoken** is a open source tokenizing library by OpenAI. It's source code available here: https://github.com/openai/gpt-2/blob/master/src/encoder.py 


In [ ]:
import tiktoken
bpe=tiktoken.get_encoding("gpt2")

In [2]:
sample_text='''Hello this is SubuGPT...<|endoftext|> How can I help you'''
encoded_text=bpe.encode(sample_text,allowed_special={"<|endoftext|>"})
print(encoded_text)
decoded_text=bpe.decode(encoded_text)
print(decoded_text)

[15496, 428, 318, 3834, 84, 38, 11571, 986, 50256, 1374, 460, 314, 1037, 345]
Hello this is SubuGPT...<|endoftext|> How can I help you


- In the code segment given below you can see, how it breaks the unknown word "SubuGPT" to  assigns tokens to the segments.

In [3]:
decoded_sample=bpe.encode("SubuGPT",allowed_special={"<|endoftext|>"})
print(decoded_sample)

[7004, 84, 38, 11571]


In [4]:
for i in decoded_sample:
    print(f'{bpe.decode([i])} ')

Sub 
u 
G 
PT 


In [5]:
list= []
for i in encoded_text:
    list.append(bpe.decode([i]))
print(list)

['Hello', ' this', ' is', ' Sub', 'u', 'G', 'PT', '...', '<|endoftext|>', ' How', ' can', ' I', ' help', ' you']


## Creating Dataset for training using Sliding Window
- The main goal of GPT as we know is next word prediction.
- So we need to create a dataset where the input label is the current sequence and output is the next sequence of words.
- We will follow a **sliding window** approach for that.

**Stride:**
- It is the number of steps the next input label is ahead of the current input label.
- As we increase the stride, overlapping between input labels decreases, reduces overfitting. But if the stride is more than window size,it could not cover the complete text data. 
- Hence, the optimal value of stride is the window size

<p align="center"><img src="Images/Screenshot 2025-06-16 110350.png" width="" height=""></p>

**Steps for Creating Dataset class**
1. Initialise features and labels
2. Tokenize text
3. Check if there is sufficient tokens
4. Generate features and labels using sliding window
5. Convert feature and labels to torch tensor

- NOTE: Our GPTDataset must be inherited to `Dataset` class



In [21]:
import torch
from torch.utils.data import Dataset, DataLoader

In [14]:
class GPTDataset(Dataset):
    def __init__(self,txt,window_size,tokeniser,stride):
        # Initialise features and labels
        self.features=[]
        self.labels=[]
        # Tokenize text
        encoded_text=tokeniser.encode(txt,allowed_special={"<|endoftext|>"})
        token_size=len(encoded_text)
        # Check if there is sufficient tokens
        assert token_size>window_size,"No of tokens must be greater than window_size"
        # Generate features and labels using sliding window
        for i in range(0,token_size-window_size,stride):
            input_chunk=encoded_text[i:i+window_size]
            output_chunk=encoded_text[i+1:i+window_size+1]
            self.features.append(input_chunk)
            self.labels.append(output_chunk)
        # Convert feature and labels to torch tensor
        self.features=torch.tensor(self.features)
        self.labels=torch.tensor(self.labels)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index],self.labels[index]
    

- Similarly, we will create `DataLoader` class for our `Dataset` class.

**Steps in Dataloader function**
1. Initiate tokenizer
2. Create Dataset from text
3. Create Dataloader from Dataset
4. Return Dataloader

In [ ]:
def create_dataloader(txt,max_length=256,shuffle=True,num_workers=0,batch_size=4,stride=128,drop_last=True):
    # Initiate tokenizer
    bpt=tiktoken.get_encoding("gpt2")
    # Create Dataset from text
    dataset=GPTDataset(txt,max_length,bpt,stride)
    # Create Dataloader from Dataset
    dataloader=DataLoader(
        dataset=dataset,
        shuffle=shuffle,
        batch_size=batch_size,
        drop_last=drop_last,
        num_workers=num_workers
    )
    # Return Dataloader
    return dataloader

In [23]:
with open("Dataset/the-verdict.txt",mode="r",encoding="utf-8") as file:
    text_data=file.read()

In [24]:
sample_data=create_dataloader(text_data, batch_size=8, max_length=4, stride=4, shuffle=False)

In [25]:
data_iter=iter(sample_data)
input1,labels1=next(data_iter)
input1,labels1

(tensor([[   40,   367,  2885,  1464],
         [ 1807,  3619,   402,   271],
         [10899,  2138,   257,  7026],
         [15632,   438,  2016,   257],
         [  922,  5891,  1576,   438],
         [  568,   340,   373,   645],
         [ 1049,  5975,   284,   502],
         [  284,  3285,   326,    11]]),
 tensor([[  367,  2885,  1464,  1807],
         [ 3619,   402,   271, 10899],
         [ 2138,   257,  7026, 15632],
         [  438,  2016,   257,   922],
         [ 5891,  1576,   438,   568],
         [  340,   373,   645,  1049],
         [ 5975,   284,   502,   284],
         [ 3285,   326,    11,   287]]))

In [26]:
type(input1)

torch.Tensor

## Embeddings
- The concept of converting tokens to vector format is called as **embedding**.
- For our GPT, we need to convert tokens to embeddings, since embeddings are trainable values, which can updated during training.
- There are 2 types embedding:
    1. **Token Embeddings:**  It defines the standard meaning of a token, just like a dictionary mentions the standard meaning of a word. Tokens with similar meaning has more **cosine similarity** between their embeddings.
    2. **Positional Embeddings:** It captures the position of the word in a sentence. It is necessary since, self attention mechanisms depend on position of the word in a sentence.
- There are 3 types of postional embedding: Absolute, Relative and Rotary positional embeddings.
- Since OpenAI's GPT uses Absolute Positional embeddings, we will use for our model.
- **Absolute Positional embedding** captures the position of the token in a sequence.

**Creating Token Embeddings**
- We follow 2 steps:
    1. Create token embedding layer of dimention `(vocab_size,output_dim)`
    2. Passing input to embedding layer to given token embeddings
- `vocab_size`: It is the number unique tokens in a vocabulary or language. 

In [ ]:
torch.manual_seed(1)
# Creating token embedding layer
vocab_size_1 = 6
output_dim_1 = 3

embedding_layer = torch.nn.Embedding(vocab_size_1,output_dim_1)
print(f"Token emddings: {embedding_layer.weight}")
print(f'Shape: {embedding_layer.weight.shape}')

# Passing inputs to token embedding layer
sample_input=torch.tensor([[1,3],[5,2]])
sample_token_embeddings=embedding_layer(sample_input)
print(sample_token_embeddings.shape)

Token emddings: Parameter containing:
tensor([[-1.5256, -0.7502,  0.6995],
        [ 0.1991,  0.8657,  0.2444],
        [-0.6629,  0.8073,  0.4391],
        [ 1.1712, -2.2456, -1.4465],
        [ 0.0612, -0.6177, -0.7981],
        [-0.1316, -0.7984,  0.3357]], requires_grad=True)
Shape: torch.Size([6, 3])
torch.Size([2, 2, 3])


- The BPE has a vocabulary size of 50,257 and GPT-2 encodes these tokens to a 256-dimentional embedding.

In [57]:
torch.manual_seed(123)
vocab_size=50257
output_dim=256
token_embedding_layer=torch.nn.Embedding(vocab_size,output_dim)
print(token_embedding_layer.weight.shape)

torch.Size([50257, 256])


In [59]:
token_embeddings=token_embedding_layer(input1)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


**Creating Positional Embeddings**
- Here we follow 2 steps:
    1. Create positional embedding layer of dimention `(window_size,output_dim)`
    2. Passing range of values from 0 to `window_size`-1 to positional embedding layer

In [65]:
# Creating postional embedding layer
window_size=input1.shape[1]
pos_embedding_layer=torch.nn.Embedding(window_size,output_dim)
# Passing range of postions to layer
pos_embeddings=pos_embedding_layer(torch.arange(window_size))
print(pos_embeddings.shape)

torch.Size([4, 256])


- The final embedding that we get is by adding token embedding and positional embedding

In [66]:
embeddings=token_embeddings+pos_embeddings
print(embeddings.shape)

torch.Size([8, 4, 256])


In [70]:
sample_text="Hello! How are you?"
token_ids=bpe.encode(sample_text)
print(token_ids)
tokens=[bpe.decode([id]) for id in token_ids]
print(tokens)

[15496, 0, 1374, 389, 345, 30]
['Hello', '!', ' How', ' are', ' you', '?']


## Summary
<p align="center"><img src="Images/Screenshot 2025-06-16 170133.png" width="" height=""></p>